## 1. Import libraries

In [ ]:
#customer segmentation based on purchase behaviour
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 2. Create a sample customer dataset
This small example dataset is created in the notebook. A separate CSV file is not required.

In [ ]:
df = pd.DataFrame({
    "customer_id": range(1, 16),
    "recency": [5, 8, 10, 12, 7, 35, 40, 45, 50, 38, 90, 100, 110, 120, 95],
    "frequency": [15, 12, 14, 11, 16, 7, 6, 8, 5, 7, 2, 3, 1, 2, 3],
    "monetary": [1500, 1200, 1400, 1100, 1600, 700, 650, 800, 500, 750, 200, 300, 150, 250, 350]
})

print(df.head())

## 3. Explore the data

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nSummary:")
display(df.describe())

## 4. Select RFM features

We use `recency`, `frequency`, and `monetary` for clustering. We do not use `customer_id` because it is only an identifier.

In [ ]:
X = df[["recency", "frequency", "monetary"]]

## 5. Scale the features

The RFM columns have different numerical ranges. StandardScaler puts them on comparable scales so that a large monetary value does not dominate the distance calculation.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 6. Choose K using the Elbow Method
WCSS (inertia) measures the squared distances between data points and their assigned cluster centres. Look for where the decrease starts to slow down.

In [ ]:
wcss = []

for k in range(1, 8):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    wcss.append(model.inertia_)

plt.plot(range(1, 8), wcss, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

## 7. Train the K-Means model

For this sample activity, we use K = 3 as a starting choice. In a real task, compare the elbow graph and silhouette scores before deciding.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print(df[["customer_id", "recency", "frequency", "monetary", "cluster"]])

## 8. Evaluate using Silhouette Score
The score ranges from -1 to +1. A higher score generally means clusters are better separated.

In [ ]:
score = silhouette_score(X_scaled, df["cluster"])
print("Silhouette Score:", score)

## 9. Understand the customer groups
The averages help us describe the purchasing behaviour in each cluster. Cluster numbers are identifiers; they do not mean good or bad.

In [ ]:
cluster_summary = df.groupby("cluster")[["recency", "frequency", "monetary"]].mean()
display(cluster_summary)

print("Customers in each cluster:")
print(df["cluster"].value_counts().sort_index())

## 10. Visualize the clusters
This simple plot compares Recency and Monetary value. The colours show the cluster assigned by K-Means.

In [ ]:
plt.scatter(
    df["recency"],
    df["monetary"],
    c=df["cluster"],
    cmap="viridis",
    s=80
)
plt.xlabel("Recency")
plt.ylabel("Monetary")
plt.title("Customer Clusters")
plt.show()

## 11. Explain in Your Own Words

1. What does RFM stand for?
RFM stands for Recency, Frequency, and Monetary.

2. Why do we use Recency, Frequency and Monetary features?
We use these features to understand how recently customers purchased, how often they purchased, and how much money they spent.

3. Why do we remove the customer ID column before clustering?
Customer ID is only an identification number. It does not help us understand customer behaviour.

4. Why is feature scaling important for K-Means?
Feature scaling puts all features on a similar scale so that larger values do not dominate the clustering.

5. What does the Elbow Method help us choose?
The Elbow Method helps us choose the best number of clusters (K).

6. What does the Silhouette Score tell us?
It tells us how well customers fit into their own clusters and how separate the clusters are from each other.

7. What does a customer cluster represent?
A customer cluster represents a group of customers who have similar purchasing behaviour.